In [ ]:
# ============================================================
# Figure 4
# R-loop program reshapes malignant cell - TME interactions
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import mannwhitneyu
from sklearn.preprocessing import StandardScaler

# ============================================================
# 0. Paths
# ============================================================

PROJECT_DIR = "/cityu/htcc/zliu-39/Test_R/R_loop_HCC"

FIG1_DIR = os.path.join(PROJECT_DIR, "01_Figure1_python")
FIG2_DIR = os.path.join(PROJECT_DIR, "02_Figure2_python")
FIG3_DIR = os.path.join(PROJECT_DIR, "03_Figure3_python")
FIG4_DIR = os.path.join(PROJECT_DIR, "04_Figure4_python")

os.makedirs(FIG4_DIR, exist_ok=True)

ADATA_PATH = os.path.join(FIG1_DIR, "Step10_adata_all_final_celltype.h5ad")

FIG3_ADATA_MAL_PATH = os.path.join(
    FIG3_DIR,
    "Figure3_malignant_cells_with_function_scores.h5ad"
)

# ============================================================
# 1. Plot settings
# ============================================================

mpl.rcParams["font.family"] = "DejaVu Sans"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["axes.unicode_minus"] = False

def save_fig(fig, name, w=5, h=4):
    fig.set_size_inches(w, h)
    for ext in ["png", "pdf", "svg"]:
        fig.savefig(
            os.path.join(FIG4_DIR, f"{name}.{ext}"),
            dpi=300,
            bbox_inches="tight",
            transparent=True
        )
    plt.close(fig)
    print("Saved:", name)

# ============================================================
# 2. Load data
# ============================================================

adata = sc.read_h5ad(ADATA_PATH)
adata_mal = sc.read_h5ad(FIG3_ADATA_MAL_PATH)

print(adata)
print(adata_mal)

print("adata.obs columns:")
print(adata.obs.columns.tolist())

print("adata_mal.obs columns:")
print(adata_mal.obs.columns.tolist())

In [ ]:
# ============================================================
# Figure4A
# TME composition across samples
# ============================================================

# 自动识别 sample 列
possible_sample_cols = ["sample", "Sample", "sample_id", "orig.ident", "dataset"]
sample_col = None

for col in possible_sample_cols:
    if col in adata.obs.columns:
        sample_col = col
        break

if sample_col is None:
    raise ValueError("No sample column found in adata.obs.")

print("Using sample column:", sample_col)

# 自动识别 cell type 列
possible_celltype_cols = [
    "celltype",
    "cell_type",
    "celltype_manual",
    "celltype_manual2",
    "celltype_final",
    "major_celltype"
]

celltype_col = None

for col in possible_celltype_cols:
    if col in adata.obs.columns:
        celltype_col = col
        break

if celltype_col is None:
    raise ValueError("No cell type column found in adata.obs.")

print("Using celltype column:", celltype_col)

# 计算每个样本中的细胞比例
comp_df = (
    adata.obs
    .groupby([sample_col, celltype_col])
    .size()
    .reset_index(name="n")
)

comp_df["total"] = comp_df.groupby(sample_col)["n"].transform("sum")
comp_df["proportion"] = comp_df["n"] / comp_df["total"] * 100

comp_wide = comp_df.pivot(
    index=sample_col,
    columns=celltype_col,
    values="proportion"
).fillna(0)

# 按 malignant cell 比例排序
malignant_candidates = [
    "Malignant cell",
    "Malignant",
    "Malignant_Epithelial",
    "malignant cell",
    "malignant"
]

mal_col = None
for x in malignant_candidates:
    if x in comp_wide.columns:
        mal_col = x
        break

if mal_col is not None:
    comp_wide = comp_wide.sort_values(mal_col, ascending=False)

comp_wide.to_csv(
    os.path.join(FIG4_DIR, "Figure4A_TME_composition_per_sample.csv")
)

# 颜色
celltype_colors = {
    "Malignant cell": "#D73027",
    "Malignant": "#D73027",
    "Malignant_Epithelial": "#D73027",
    "T cell": "#4575B4",
    "T_NK": "#4575B4",
    "CD4 T": "#74ADD1",
    "CD8 T": "#313695",
    "Treg": "#984EA3",
    "TAM": "#4DAF4A",
    "Myeloid": "#4DAF4A",
    "B cell": "#F781BF",
    "B_Plasma": "#F781BF",
    "CAF": "#A65628",
    "Fibroblast": "#A65628",
    "TEC": "#999999",
    "Endothelial": "#999999",
    "HPC-like": "#FDB462",
    "unclassified": "#BDBDBD"
}

colors = [
    celltype_colors.get(x, "#BDBDBD")
    for x in comp_wide.columns
]

fig, ax = plt.subplots(figsize=(12, 4.5))

bottom = np.zeros(comp_wide.shape[0])
x = np.arange(comp_wide.shape[0])

for ct, color in zip(comp_wide.columns, colors):
    ax.bar(
        x,
        comp_wide[ct].values,
        bottom=bottom,
        color=color,
        label=ct,
        width=0.85
    )
    bottom += comp_wide[ct].values

ax.set_ylabel("Cell proportion (%)", fontsize=11)
ax.set_xlabel("")
ax.set_ylim(0, 100)
ax.set_xticks(x)
ax.set_xticklabels(comp_wide.index.astype(str), rotation=90, fontsize=7)

ax.set_title(
    "Tumor microenvironment composition across samples",
    fontsize=14,
    fontweight="bold",
    pad=10
)

ax.legend(
    frameon=False,
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=9
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

save_fig(
    fig,
    "Figure4A_TME_composition_across_samples",
    w=12,
    h=4.5
)

In [ ]:
# ============================================================
# Figure4B
# TME composition associated with R-loop program enrichment
# Effect size / trend annotation + wider spacing between paired boxes
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import mannwhitneyu, ttest_ind

# ------------------------------------------------------------
# 1. 自动识别 adata_mal 里的 sample 列
# ------------------------------------------------------------

possible_sample_cols = ["sample", "Sample", "sample_id", "orig.ident", "dataset"]
sample_col_mal = None

for col in possible_sample_cols:
    if col in adata_mal.obs.columns:
        sample_col_mal = col
        break

if sample_col_mal is None:
    print("adata_mal.obs columns:")
    print(adata_mal.obs.columns.tolist())
    raise ValueError("No sample column found in adata_mal.obs.")

print("Using malignant sample column:", sample_col_mal)

# ------------------------------------------------------------
# 2. 计算每个样本中 R-loop-program-high 的比例
# ------------------------------------------------------------

rloop_prop = (
    adata_mal.obs
    .groupby([sample_col_mal, "Rloop_program_group"])
    .size()
    .reset_index(name="n")
)

rloop_prop["total"] = rloop_prop.groupby(sample_col_mal)["n"].transform("sum")
rloop_prop["proportion"] = rloop_prop["n"] / rloop_prop["total"] * 100

rloop_wide = rloop_prop.pivot(
    index=sample_col_mal,
    columns="Rloop_program_group",
    values="proportion"
).fillna(0)

if "R-loop-program-high" not in rloop_wide.columns:
    raise ValueError("R-loop-program-high not found in rloop_wide.")

rloop_wide["Rloop_sample_group"] = np.where(
    rloop_wide["R-loop-program-high"] >= rloop_wide["R-loop-program-high"].median(),
    "R-loop-high-enriched",
    "R-loop-low-enriched"
)

# ------------------------------------------------------------
# 3. 使用 Figure4A 已经生成的 comp_wide
# comp_wide: index = sample, columns = cell types, values = proportion
# ------------------------------------------------------------

if "comp_wide" not in globals():
    raise ValueError("comp_wide not found. Please run Figure4A code first.")

comp_wide2 = comp_wide.copy()
comp_wide2.index = comp_wide2.index.astype(str)

rloop_wide2 = rloop_wide.copy()
rloop_wide2.index = rloop_wide2.index.astype(str)

common_samples = comp_wide2.index.intersection(rloop_wide2.index)

print("Number of common samples:", len(common_samples))

if len(common_samples) == 0:
    print("comp_wide samples:")
    print(comp_wide2.index[:10].tolist())
    print("rloop_wide samples:")
    print(rloop_wide2.index[:10].tolist())
    raise ValueError("No common samples between comp_wide and rloop_wide.")

tme_for_test = comp_wide2.loc[common_samples].copy()
tme_for_test["Rloop_sample_group"] = rloop_wide2.loc[
    common_samples,
    "Rloop_sample_group"
]

# ------------------------------------------------------------
# 4. 去除 malignant，只保留 TME 细胞类型
# ------------------------------------------------------------

drop_cols = [
    x for x in tme_for_test.columns
    if ("Malignant" in str(x)) or (str(x) == "Rloop_sample_group")
]

plot_cols = [
    x for x in tme_for_test.columns
    if x not in drop_cols
]

major_cols = [
    x for x in plot_cols
    if tme_for_test[x].mean() > 1
]

print("Major TME cell types:", major_cols)

plot_long = tme_for_test.melt(
    id_vars="Rloop_sample_group",
    value_vars=major_cols,
    var_name="Cell type",
    value_name="Proportion"
)

plot_long["Rloop_sample_group"] = pd.Categorical(
    plot_long["Rloop_sample_group"],
    categories=["R-loop-low-enriched", "R-loop-high-enriched"],
    ordered=True
)

plot_long["Cell type"] = pd.Categorical(
    plot_long["Cell type"],
    categories=major_cols,
    ordered=True
)

plot_long.to_csv(
    os.path.join(
        FIG4_DIR,
        "Figure4B_TME_composition_Rloop_sample_group_effect_size_data.csv"
    ),
    index=False
)

# ------------------------------------------------------------
# 5. 计算 effect size / trend
# ------------------------------------------------------------

stat_rows = []

for ct in major_cols:
    vals_low = plot_long.loc[
        (plot_long["Cell type"] == ct) &
        (plot_long["Rloop_sample_group"] == "R-loop-low-enriched"),
        "Proportion"
    ].dropna()

    vals_high = plot_long.loc[
        (plot_long["Cell type"] == ct) &
        (plot_long["Rloop_sample_group"] == "R-loop-high-enriched"),
        "Proportion"
    ].dropna()

    mean_low = vals_low.mean()
    mean_high = vals_high.mean()
    median_low = vals_low.median()
    median_high = vals_high.median()

    delta_mean = mean_high - mean_low
    delta_median = median_high - median_low

    if len(vals_low) > 1 and len(vals_high) > 1:
        _, p_wilcox = mannwhitneyu(vals_low, vals_high, alternative="two-sided")
        _, p_ttest = ttest_ind(vals_low, vals_high, equal_var=False)
    else:
        p_wilcox = np.nan
        p_ttest = np.nan

    pooled_sd = np.sqrt(
        ((len(vals_low) - 1) * vals_low.var(ddof=1) +
         (len(vals_high) - 1) * vals_high.var(ddof=1)) /
        (len(vals_low) + len(vals_high) - 2)
    ) if (len(vals_low) > 1 and len(vals_high) > 1) else np.nan

    cohen_d = delta_mean / pooled_sd if pooled_sd and pooled_sd > 0 else np.nan

    stat_rows.append({
        "Cell type": ct,
        "n_low": len(vals_low),
        "n_high": len(vals_high),
        "mean_low": mean_low,
        "mean_high": mean_high,
        "median_low": median_low,
        "median_high": median_high,
        "delta_mean_high_minus_low": delta_mean,
        "delta_median_high_minus_low": delta_median,
        "cohen_d": cohen_d,
        "p_wilcox": p_wilcox,
        "p_ttest_welch": p_ttest
    })

stat_df = pd.DataFrame(stat_rows)

stat_df.to_csv(
    os.path.join(
        FIG4_DIR,
        "Figure4B_TME_composition_effect_size.csv"
    ),
    index=False
)

print(stat_df)

# ------------------------------------------------------------
# 6. 绘图：手动绘制 boxplot，让两个 paired boxes 间距更大
# ------------------------------------------------------------

group_order = ["R-loop-low-enriched", "R-loop-high-enriched"]

palette = {
    "R-loop-low-enriched": "#ff7f0e",
    "R-loop-high-enriched": "#1f77b4"
}

fig, ax = plt.subplots(figsize=(9.8, 5.4))

base_x = np.arange(len(major_cols))

box_width = 0.18
offset = 0.17     # 这个越大，两个 paired boxes 间距越大
point_jitter = 0.035

for i, ct in enumerate(major_cols):
    for group in group_order:
        vals = plot_long.loc[
            (plot_long["Cell type"] == ct) &
            (plot_long["Rloop_sample_group"] == group),
            "Proportion"
        ].dropna().values

        if group == "R-loop-low-enriched":
            xpos = base_x[i] - offset
        else:
            xpos = base_x[i] + offset

        bp = ax.boxplot(
            vals,
            positions=[xpos],
            widths=box_width,
            patch_artist=True,
            showfliers=False,
            boxprops=dict(
                facecolor=palette[group],
                edgecolor="black",
                linewidth=1.15,
                alpha=0.85
            ),
            medianprops=dict(
                color="black",
                linewidth=1.2
            ),
            whiskerprops=dict(
                color="black",
                linewidth=1.0
            ),
            capprops=dict(
                color="black",
                linewidth=1.0
            )
        )

        # 添加散点
        rng = np.random.default_rng(42 + i)
        x_jitter = rng.normal(
            loc=xpos,
            scale=point_jitter,
            size=len(vals)
        )

        ax.scatter(
            x_jitter,
            vals,
            s=18,
            color=palette[group],
            alpha=0.65,
            linewidths=0,
            zorder=3
        )

# ------------------------------------------------------------
# 7. 添加 effect size / trend 标注
# ------------------------------------------------------------

all_y = plot_long["Proportion"].dropna()
y_min = min(-2, all_y.min() - 2)
y_max = all_y.max()
y_range = y_max - y_min

for i, ct in enumerate(major_cols):
    df_ct = plot_long[plot_long["Cell type"] == ct]
    y = df_ct["Proportion"].max()

    delta = stat_df.loc[
        stat_df["Cell type"] == ct,
        "delta_mean_high_minus_low"
    ].values[0]

    p_val = stat_df.loc[
        stat_df["Cell type"] == ct,
        "p_ttest_welch"
    ].values[0]

    if delta > 0:
        label = f"Δ={delta:+.1f}% ↑"
    elif delta < 0:
        label = f"Δ={delta:+.1f}% ↓"
    else:
        label = f"Δ={delta:+.1f}%"

    if not np.isnan(p_val) and p_val < 0.1:
        label = label + "\ntrend"

    y_text = y + 0.07 * y_range

    ax.text(
        base_x[i],
        y_text,
        label,
        ha="center",
        va="bottom",
        fontsize=9.5,
        fontweight="bold"
    )

# ------------------------------------------------------------
# 8. 图例和整体美化
# ------------------------------------------------------------

from matplotlib.patches import Patch

legend_handles = [
    Patch(
        facecolor=palette["R-loop-low-enriched"],
        edgecolor="black",
        label="R-loop-low-enriched",
        alpha=0.85
    ),
    Patch(
        facecolor=palette["R-loop-high-enriched"],
        edgecolor="black",
        label="R-loop-high-enriched",
        alpha=0.85
    )
]

ax.legend(
    handles=legend_handles,
    frameon=False,
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=10
)

ax.set_xticks(base_x)
ax.set_xticklabels(
    major_cols,
    rotation=45,
    ha="right",
    rotation_mode="anchor",
    fontsize=11
)

ax.set_xlim(-0.6, len(major_cols) - 0.4)
ax.set_ylim(y_min, y_max + 0.25 * y_range)

ax.set_ylabel("Cell proportion (%)", fontsize=12)
ax.set_xlabel("")

ax.set_title(
    "TME composition associated with R-loop-program enrichment",
    fontsize=15,
    fontweight="bold",
    pad=12
)

ax.tick_params(axis="y", labelsize=11)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

save_fig(
    fig,
    "Figure4B_TME_composition_Rloop_sample_group_effect_size_wider_spacing",
    w=9.8,
    h=5.4
)

In [ ]:
# ============================================================
# Figure4C refined
# Immune-interaction molecules in malignant R-loop groups
# Compact two-column dotplot
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp
import os

# ------------------------------------------------------------
# 1. Define LR-related genes
# ------------------------------------------------------------

lr_genes = {
    "Immune-activating": [
        "CXCL9", "CXCL10", "CXCL16", "CCL5",
        "HLA-A", "HLA-B", "B2M", "TAP1", "ICAM1", "TNFSF10"
    ],
    "Immune-suppressive": [
        "CD274", "PDCD1LG2", "LGALS9", "TGFB1",
        "SPP1", "MIF", "VEGFA", "NECTIN2", "CD47", "PVR"
    ]
}

genes_use = []
gene_category = {}

for cat, genes in lr_genes.items():
    for g in genes:
        if g in adata_mal.var_names:
            genes_use.append(g)
            gene_category[g] = cat

genes_use = list(dict.fromkeys(genes_use))

print("Genes used:", genes_use)

if len(genes_use) == 0:
    raise ValueError("No LR-related genes found in adata_mal.var_names.")

# ------------------------------------------------------------
# 2. Extract expression
# ------------------------------------------------------------

X = adata_mal[:, genes_use].X
if sp.issparse(X):
    X = X.toarray()

expr_df = pd.DataFrame(
    X,
    index=adata_mal.obs_names,
    columns=genes_use
)

expr_df["Rloop_program_group"] = adata_mal.obs["Rloop_program_group"].values

rows = []

for group in ["R-loop-program-low", "R-loop-program-high"]:
    df_g = expr_df[expr_df["Rloop_program_group"] == group]

    for gene in genes_use:
        vals = df_g[gene].values

        rows.append({
            "Group": group,
            "Gene": gene,
            "Category": gene_category[gene],
            "Mean expression": np.mean(vals),
            "Fraction expressed": np.mean(vals > 0) * 100
        })

dot_df = pd.DataFrame(rows)

# ------------------------------------------------------------
# 3. Scale expression per gene
# ------------------------------------------------------------

dot_df["Scaled mean expression"] = dot_df.groupby("Gene")["Mean expression"].transform(
    lambda x: (x - x.mean()) / (x.std() + 1e-8)
)

dot_df["Scaled mean expression"] = dot_df["Scaled mean expression"].clip(-1, 1)

# ------------------------------------------------------------
# 4. Gene order
# ------------------------------------------------------------

gene_order = (
    pd.DataFrame({
        "Gene": genes_use,
        "Category": [gene_category[g] for g in genes_use]
    })
    .sort_values(["Category", "Gene"])
    ["Gene"]
    .tolist()
)

# 让图上从上到下更自然
gene_order = gene_order[::-1]

dot_df["Gene"] = pd.Categorical(
    dot_df["Gene"],
    categories=gene_order,
    ordered=True
)

group_order = ["R-loop-program-low", "R-loop-program-high"]

group_label_map = {
    "R-loop-program-low": "Low",
    "R-loop-program-high": "High"
}

dot_df["Group_short"] = dot_df["Group"].map(group_label_map)

dot_df["Group_short"] = pd.Categorical(
    dot_df["Group_short"],
    categories=["Low", "High"],
    ordered=True
)

# 手动 x 坐标，让两列靠近
x_map = {
    "Low": 0,
    "High": 0.65
}

dot_df["x"] = dot_df["Group_short"].map(x_map).astype(float)
dot_df["y"] = dot_df["Gene"].cat.codes.astype(float)

# ------------------------------------------------------------
# 5. Plot
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(4.8, 5.8))

# 背景浅灰横线
for y in sorted(dot_df["y"].unique()):
    ax.axhline(
        y=y,
        color="#EAEAEA",
        linewidth=0.8,
        zorder=0
    )

scatter = ax.scatter(
    dot_df["x"],
    dot_df["y"],
    s=dot_df["Fraction expressed"] * 5.5,
    c=dot_df["Scaled mean expression"],
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    edgecolor="black",
    linewidth=0.35,
    alpha=0.95,
    zorder=3
)

# ------------------------------------------------------------
# 6. Axis
# ------------------------------------------------------------

ax.set_xticks([0, 0.65])
ax.set_xticklabels(
    ["R-loop\nprogram-low", "R-loop\nprogram-high"],
    fontsize=11,
    rotation=0
)

ax.set_xlim(-0.25, 0.90)

ax.set_yticks(range(len(gene_order)))
ax.set_yticklabels(gene_order, fontsize=10)

ax.set_xlabel("")
ax.set_ylabel("")

ax.set_title(
    "Immune-interaction molecules\nin malignant R-loop groups",
    fontsize=14,
    fontweight="bold",
    pad=10
)

ax.tick_params(axis="both", length=0)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# ------------------------------------------------------------
# 7. Colorbar
# ------------------------------------------------------------

cbar = plt.colorbar(
    scatter,
    ax=ax,
    fraction=0.045,
    pad=0.035
)

cbar.set_label(
    "Scaled mean\nexpression",
    fontsize=10
)

cbar.ax.tick_params(labelsize=9)

# ------------------------------------------------------------
# 8. Size legend
# ------------------------------------------------------------

size_values = [10, 20, 30, 40]

size_handles = [
    ax.scatter(
        [],
        [],
        s=x * 5.5,
        color="grey",
        edgecolor="black",
        linewidth=0.35,
        alpha=0.85
    )
    for x in size_values
]

size_legend = ax.legend(
    size_handles,
    [str(x) for x in size_values],
    title="Fraction\nexpressed (%)",
    frameon=False,
    bbox_to_anchor=(1.28, 0.48),
    loc="center left",
    fontsize=9,
    title_fontsize=10
)

ax.add_artist(size_legend)

# ------------------------------------------------------------
# 9. Save
# ------------------------------------------------------------

dot_df.to_csv(
    os.path.join(FIG4_DIR, "Figure4C_LR_related_gene_dotplot_refined_data.csv"),
    index=False
)

save_fig(
    fig,
    "Figure4C_LR_related_gene_dotplot_refined",
    w=4.8,
    h=5.8
)

In [ ]:
# ============================================================
# Figure4D
# T cell functional states associated with R-loop program
# with significance
# ============================================================

import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu

# ------------------------------------------------------------
# 1. 固定细胞类型列
# ------------------------------------------------------------

celltype_col = "celltype"

print(adata.obs[celltype_col].value_counts())

# ------------------------------------------------------------
# 2. 自动找到与 rloop_wide 匹配的样本列
# ------------------------------------------------------------

rloop_wide2 = rloop_wide.copy()
rloop_wide2.index = rloop_wide2.index.astype(str)

best_sample_col = None
best_overlap = 0

for col in adata.obs.columns:
    vals = adata.obs[col].astype(str).unique()
    overlap = len(set(vals).intersection(set(rloop_wide2.index)))

    if overlap > best_overlap:
        best_overlap = overlap
        best_sample_col = col

if best_sample_col is None or best_overlap == 0:
    print("adata.obs columns:")
    print(adata.obs.columns.tolist())
    print("rloop_wide index examples:")
    print(rloop_wide2.index[:10].tolist())
    raise ValueError("No sample column in adata.obs matches rloop_wide index.")

sample_col = best_sample_col

print("Using sample column:", sample_col)
print("Matched sample number:", best_overlap)

# ------------------------------------------------------------
# 3. T cell functional gene sets
# ------------------------------------------------------------

immune_gene_sets = {
    "Cytotoxicity": [
        "GZMB", "GZMA", "GZMK", "PRF1", "NKG7", "GNLY", "IFNG"
    ],
    "Exhaustion": [
        "PDCD1", "CTLA4", "LAG3", "HAVCR2", "TIGIT", "TOX", "ENTPD1"
    ],
    "Treg": [
        "FOXP3", "IL2RA", "CTLA4", "IKZF2", "TNFRSF18"
    ]
}

# ------------------------------------------------------------
# 4. 选择 T cell
# ------------------------------------------------------------

adata_tcell = adata[
    adata.obs[celltype_col].astype(str).isin(["T cell"])
].copy()

print(adata_tcell)
print(adata_tcell.obs[celltype_col].value_counts())

# ------------------------------------------------------------
# 5. 计算 signature scores
# ------------------------------------------------------------

for sig, genes in immune_gene_sets.items():
    genes_use = [g for g in genes if g in adata_tcell.var_names]

    print(sig, len(genes_use), genes_use)

    if len(genes_use) >= 3:
        sc.tl.score_genes(
            adata_tcell,
            gene_list=genes_use,
            score_name=f"{sig}_score",
            use_raw=False
        )

score_cols_immune = [
    f"{x}_score"
    for x in immune_gene_sets.keys()
    if f"{x}_score" in adata_tcell.obs.columns
]

signature_order = [
    x.replace("_score", "")
    for x in score_cols_immune
]

print("Signature order:", signature_order)

# ------------------------------------------------------------
# 6. 加入 R-loop sample group
# ------------------------------------------------------------

adata_tcell.obs[sample_col] = adata_tcell.obs[sample_col].astype(str)

adata_tcell.obs["Rloop_sample_group"] = adata_tcell.obs[sample_col].map(
    rloop_wide2["Rloop_sample_group"].to_dict()
)

print(adata_tcell.obs["Rloop_sample_group"].value_counts(dropna=False))

# ------------------------------------------------------------
# 7. 整理绘图数据
# ------------------------------------------------------------

plot_df = adata_tcell.obs[
    ["Rloop_sample_group", celltype_col] + score_cols_immune
].dropna().copy()

plot_long_d = plot_df.melt(
    id_vars=["Rloop_sample_group", celltype_col],
    value_vars=score_cols_immune,
    var_name="Signature",
    value_name="Score"
)

plot_long_d["Signature"] = plot_long_d["Signature"].str.replace(
    "_score",
    "",
    regex=False
)

plot_long_d = plot_long_d[
    plot_long_d["Rloop_sample_group"].isin(
        ["R-loop-low-enriched", "R-loop-high-enriched"]
    )
].copy()

plot_long_d["Signature"] = pd.Categorical(
    plot_long_d["Signature"],
    categories=signature_order,
    ordered=True
)

plot_long_d["Rloop_sample_group"] = pd.Categorical(
    plot_long_d["Rloop_sample_group"],
    categories=["R-loop-low-enriched", "R-loop-high-enriched"],
    ordered=True
)

# ------------------------------------------------------------
# 8. 统计检验
# ------------------------------------------------------------

group_order = ["R-loop-low-enriched", "R-loop-high-enriched"]

palette = {
    "R-loop-high-enriched": "#1f77b4",
    "R-loop-low-enriched": "#ff7f0e"
}

def p_to_star(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

stat_rows = []

for sig in signature_order:
    vals_low = plot_long_d.loc[
        (plot_long_d["Signature"] == sig) &
        (plot_long_d["Rloop_sample_group"] == "R-loop-low-enriched"),
        "Score"
    ].dropna()

    vals_high = plot_long_d.loc[
        (plot_long_d["Signature"] == sig) &
        (plot_long_d["Rloop_sample_group"] == "R-loop-high-enriched"),
        "Score"
    ].dropna()

    if len(vals_low) > 0 and len(vals_high) > 0:
        _, p = mannwhitneyu(vals_low, vals_high, alternative="two-sided")
    else:
        p = np.nan

    stat_rows.append({
        "Signature": sig,
        "n_low": len(vals_low),
        "n_high": len(vals_high),
        "median_low": vals_low.median(),
        "median_high": vals_high.median(),
        "delta_median_high_minus_low": vals_high.median() - vals_low.median(),
        "p_value": p,
        "star": p_to_star(p) if not np.isnan(p) else "NA"
    })

stat_df_d = pd.DataFrame(stat_rows)

stat_df_d.to_csv(
    os.path.join(FIG4_DIR, "Figure4D_Tcell_function_statistics.csv"),
    index=False
)

print(stat_df_d)

# ------------------------------------------------------------
# 9. 绘图
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7.8, 4.8))

sns.violinplot(
    data=plot_long_d,
    x="Signature",
    y="Score",
    hue="Rloop_sample_group",
    order=signature_order,
    hue_order=group_order,
    palette=palette,
    split=True,
    inner=None,
    cut=0,
    linewidth=1,
    ax=ax
)

sns.pointplot(
    data=plot_long_d,
    x="Signature",
    y="Score",
    hue="Rloop_sample_group",
    order=signature_order,
    hue_order=group_order,
    palette=palette,
    dodge=0.25,
    estimator=np.median,
    errorbar=None,
    markers=".",
    markersize=7,
    linestyles="",
    ax=ax
)

handles, labels = ax.get_legend_handles_labels()

ax.legend(
    handles[:2],
    labels[:2],
    frameon=False,
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=9
)

# ------------------------------------------------------------
# 10. 添加显著性标记
# ------------------------------------------------------------

y_min, y_max = ax.get_ylim()
y_range = y_max - y_min

for i, sig in enumerate(signature_order):
    df_sig = plot_long_d[plot_long_d["Signature"] == sig]
    y = df_sig["Score"].max()

    y_bar = y + 0.07 * y_range
    y_text = y + 0.10 * y_range

    ax.plot(
        [i - 0.18, i - 0.18, i + 0.18, i + 0.18],
        [y_bar, y_text, y_text, y_bar],
        lw=1,
        color="black"
    )

    star = stat_df_d.loc[
        stat_df_d["Signature"] == sig,
        "star"
    ].values[0]

    ax.text(
        i,
        y_text + 0.01 * y_range,
        star,
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold"
    )

ax.set_ylim(y_min, y_max + 0.20 * y_range)

# ------------------------------------------------------------
# 11. 美化和保存
# ------------------------------------------------------------

ax.set_xlabel("")
ax.set_ylabel("Signature score", fontsize=11)

ax.set_title(
    "T cell functional states associated with R-loop program",
    fontsize=14,
    fontweight="bold",
    pad=10
)

plt.setp(
    ax.get_xticklabels(),
    rotation=45,
    ha="right",
    rotation_mode="anchor"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", labelsize=11)

plot_long_d.to_csv(
    os.path.join(FIG4_DIR, "Figure4D_Tcell_function_scores_Rloop_sample_group.csv"),
    index=False
)

save_fig(
    fig,
    "Figure4D_Tcell_function_scores_Rloop_sample_group_with_significance",
    w=7.8,
    h=4.8
)

In [ ]:
# ============================================================
# Figure4E
# Schematic model placeholder
# ============================================================

fig, ax = plt.subplots(figsize=(7, 3.8))

ax.text(
    0.08, 0.72,
    "R-loop-program-high\nmalignant cells",
    ha="left",
    va="center",
    fontsize=13,
    fontweight="bold",
    color="#1f77b4",
    transform=ax.transAxes
)

ax.text(
    0.08, 0.32,
    "R-loop-program-low\nmalignant cells",
    ha="left",
    va="center",
    fontsize=13,
    fontweight="bold",
    color="#ff7f0e",
    transform=ax.transAxes
)

ax.annotate(
    "",
    xy=(0.55, 0.72),
    xytext=(0.33, 0.72),
    arrowprops=dict(arrowstyle="->", lw=2, color="black"),
    xycoords=ax.transAxes,
    textcoords=ax.transAxes
)

ax.annotate(
    "",
    xy=(0.55, 0.32),
    xytext=(0.33, 0.32),
    arrowprops=dict(arrowstyle="->", lw=2, color="black"),
    xycoords=ax.transAxes,
    textcoords=ax.transAxes
)

ax.text(
    0.60, 0.72,
    "Altered immune\ncommunication",
    ha="left",
    va="center",
    fontsize=12,
    transform=ax.transAxes
)

ax.text(
    0.60, 0.32,
    "Distinct T/NK\nfunctional states",
    ha="left",
    va="center",
    fontsize=12,
    transform=ax.transAxes
)

ax.text(
    0.5, 0.05,
    "R-loop program may remodel malignant cell–TME interactions",
    ha="center",
    va="bottom",
    fontsize=12,
    fontweight="bold",
    transform=ax.transAxes
)

ax.axis("off")

save_fig(
    fig,
    "Figure4E_Schematic_model_placeholder",
    w=7,
    h=3.8
)